# 🎻 Mara's road: REINFORCE, as the libraries write it

*Policy gradient methods, Part II, Day 3. A coding exercise.*

Mara the busker is back on the same road as in the first exercise: fifteen places, a fork at every place, a day of three roads, and a few coins in her hat at every arrival, give or take one coin of luck. She keeps the same table of knobs, two rows (one for each road) and seven columns (one for each place that has a fork), and a softmax turns each column into the chances of her two roads.

In the first exercise she learned with the grasshopper's trick. She picked one knob, nudged it by `eps`, played `N` days with each table and compared the averages. It worked, and it was expensive. Every iteration played 200 days to measure the slope of **one** knob out of fourteen.

This time we ask more of the same days. Look at one day that Mara plays. She took three roads, and the day paid `G` coins. If `G` is large, those three roads were probably good ones, so we push all three up, the harder the more coins the day paid. If `G` is small, we push them up less. One day tells us something about six knobs (the two knobs of each of the three places it passed), and a hundred days tell us something about all fourteen. This idea is called **REINFORCE**.

One round of the method has four steps:

1. play `B` days with the current table;
2. for every road that Mara took, compute its **score**: how the knobs must move to make that road more likely;
3. multiply each score by its **multiplier** (for now the coins of its day) and average over the days. The result is `g`, an estimate of the gradient of the average coins;
4. move the knobs: `w = w + alpha * g`. The grasshopper's line again.

Real code does steps 2 to 4 with PyTorch's autograd, through a trick called the pseudo-loss. You will write that code in the shape that the most copied tutorials give it.

### What you build

Four short pieces, each marked with 🎯: **1.** `play_days`, a policy that plays days, **2.** `pseudo_loss`, **3.** `reinforce_step` and **4.** `reward_to_go`. No PyTorch experience is needed. Before each piece, a short section says what its names mean and shows every tool the piece uses on a small example that runs in a cell of its own, and then you write the lines marked with 🎯 in the same way. After each piece a check cell says in plain words whether it works. Then you run REINFORCE on Mara's road, turn on the extras that the popular implementations add, and compare the cost with the first exercise.

Every cell runs in a few seconds on Colab's CPU, and no GPU is needed. Run the cells from the top, one after another (click a cell and press Shift and Enter). Everywhere you see 🎯 it is your turn, and everything else runs by itself. A "Not yet." from a check is normal while a piece is unfinished, and so is a red box: read the message, fix the piece, and run the cells again.

Start with the cell below. It loads the tools and the checkers, and its code is hidden because nothing in it needs your attention.

In [ ]:
# @title Run this cell first (it loads PyTorch, the exact average coins and the checkers) { display-mode: "form" }

import inspect
from collections import namedtuple

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, NullFormatter
from torch.distributions import Categorical

BULLSEYE = "\U0001F3AF"
TINY = torch.finfo(torch.float32).eps    # the PyTorch example calls it eps: a tiny number added to a spread, so that a division never meets 0

# ======================================================================
# The exact average coins of a table of knobs.
# It draws the plots and runs the checks. The method itself never calls it.
# The luck of every arrival averages to zero, so the usual coins are all it needs.
# ======================================================================

ALL_PATHS = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]   # the eight paths, as three roads


def places_of(path):
    # the three places a path visits. The roads out of place i lead to places 2i+1 and 2i+2.
    place = 0
    places = []
    for road in path:
        place = 2 * place + 1 + road
        places.append(place)
    return places


def _table(w):
    # a table of knobs as a numpy array, whether it is a tensor with gradients or not
    if hasattr(w, "detach"):
        w = w.detach().cpu().numpy()
    return np.asarray(w, dtype=float)


def first_road_chances(w):
    # the chance of road 0 in every place, that is in every column of the table. For two knobs the softmax is 1 / (1 + exp(w2 - w1)).
    w = _table(w)
    return 1.0 / (1.0 + np.exp(w[1, :] - w[0, :]))


def path_probabilities(w):
    # the chance of each of the eight paths: multiply the chances of its three roads
    first = first_road_chances(w)
    probabilities = []
    for path in ALL_PATHS:
        place = 0
        probability = 1.0
        for road in path:
            probability = probability * (first[place] if road == 0 else 1.0 - first[place])
            place = 2 * place + 1 + road
        probabilities.append(probability)
    return np.array(probabilities)


def path_coins():
    # the usual coins of each of the eight paths, which are also their average pay
    return np.array([sum(int(COINS[p]) for p in places_of(path)) for path in ALL_PATHS])


def exact_J(w):
    # the exact average coins of a day: every path weighted by its chance
    return float(path_probabilities(w) @ path_coins())


def exact_gradient(w, h=1e-5):
    # the exact gradient of exact_J, one knob at a time (road k, place i): nudge the knob up and down by h and compare
    w = _table(w)
    g = np.zeros((2, 7))
    for i in range(7):
        for k in range(2):
            up, down = w.copy(), w.copy()
            up[k, i] += h
            down[k, i] -= h
            g[k, i] = (exact_J(up) - exact_J(down)) / (2 * h)
    return g


def most_likely_path(w):
    probabilities = path_probabilities(w)
    k = int(probabilities.argmax())
    return ALL_PATHS[k], float(probabilities[k])


def best_path():
    return ALL_PATHS[int(path_coins().argmax())]


def days_to_share(curve, B, share=0.9):
    # the days played until the exact average coins has closed `share` of the gap between the start and the best path
    start, best = curve[0], float(path_coins().max())
    target = start + share * (best - start)
    for update, value in enumerate(curve):
        if value >= target:
            return update * B
    return None


# ======================================================================
# Printing the road and some days
# ======================================================================

def show_road():
    print("place  name                 usual coins   roads out of it (road 0, road 1)")
    for i, name in enumerate(PLACES):
        roads = "%d, %d" % (2 * i + 1, 2 * i + 2) if i < 7 else "the day ends here"
        print("%5d  %-20s %11d   %s" % (i, name, COINS[i], roads))


def show_path(places):
    names = ["%s (%d)" % (PLACES[p], COINS[p]) for p in places]
    total = sum(int(COINS[p]) for p in places)
    print(" -> ".join(["Eagle Pass"] + names) + "   =   %d coins on average" % total)


def show_days(days):
    for k in range(len(days.roads)):
        arrivals = [2 * int(p) + 1 + int(r) for p, r in zip(days.places[k], days.roads[k])]
        coins = [int(c) for c in days.coins[k]]
        logps = [float(v) for v in days.logps[k]]
        print("day %d: %s" % (k + 1, " -> ".join([PLACES[0]] + [PLACES[a] for a in arrivals])))
        print("       coins %s = %d      log-probabilities %s add up to %.2f, which is the chance %.3f of these three roads together"
              % (" + ".join(str(c) for c in coins), sum(coins), "  ".join("%.2f" % v for v in logps), sum(logps), float(np.exp(sum(logps)))))


def show_draws(probs, roads, logps):
    # one line per day: the chances of its two roads, the road that was drawn, and the log-probability of that road
    for day in range(len(roads)):
        road = int(roads[day])
        print("day %d: chances %s, drew road %d, log-probability %.2f, which is the logarithm of %.1f"
              % (day + 1, [round(float(c), 2) for c in probs[day]], road, float(logps[day]), float(probs[day][road])))


# ======================================================================
# The formula written out with plain loops, and the checkers.
# Each checker calls your function on small cases and tells you what it finds.
# ======================================================================

def hand_estimate(w, places, roads, multiplier):
    # g = (1/B) * sum over the days and the roads of  multiplier * score, written out the way the formula reads.
    # The score of the road taken at a place: 1 - its chance for its knob, minus the chance of the road for the other knob.
    w = _table(w)
    chances = np.exp(w) / np.exp(w).sum(axis=0, keepdims=True)         # every column of the table adds up to 1
    places = np.asarray(places)
    roads = np.asarray(roads)
    multiplier = np.broadcast_to(np.asarray(multiplier, dtype=float), places.shape)
    g = np.zeros((2, 7))
    for b in range(len(places)):
        for t in range(3):
            place, road = int(places[b][t]), int(roads[b][t])
            for knob in (0, 1):
                score = (1.0 if knob == road else 0.0) - chances[knob][place]
                g[knob][place] += multiplier[b][t] * score
    return g / len(places)


# four days written by hand: the places where the roads were chosen, the roads, and the coins of the arrivals
_FAKE_PLACES = torch.tensor([[0, 1, 4], [0, 2, 6], [0, 1, 3], [0, 2, 5]])
_FAKE_ROADS = torch.tensor([[0, 1, 0], [1, 1, 0], [0, 0, 1], [1, 0, 1]])
_FAKE_COINS = torch.tensor([[7.0, 7.0, 7.0], [9.0, 4.0, 2.0], [7.0, 8.0, 3.0], [9.0, 3.0, 7.0]])


def _fake_days(w):
    # the record of those four days, with log-probabilities worked out from the table w by plain PyTorch
    logps = torch.log_softmax(w[:, _FAKE_PLACES], dim=0).gather(0, _FAKE_ROADS.unsqueeze(0)).squeeze(0)
    return Days(_FAKE_PLACES, _FAKE_ROADS, logps, _FAKE_COINS)


def _reference_loss(days, multiplier):
    return -(multiplier * days.logps).sum(dim=1).mean()


_passed = {}


def _capital(text):
    # a message starts with a capital letter, unless it starts with a name of the code (w, play_days, optimiser.step())
    word = text.split(" ")[0]
    if len(word) == 1 or any(ch in word for ch in "_.()="):
        return text
    return text[:1].upper() + text[1:]


def _say(n, ok, good, bad=""):
    _passed[n] = ok
    print(("Right. " + good) if ok else ("Not yet. " + _capital(bad)))


def _blanks(function):
    # the lines marked with the bullseye that still hold three dots
    try:
        lines = inspect.getsource(function).splitlines()
    except (OSError, TypeError):
        return []
    tags = []
    for line in lines:
        code, hash_sign, comment = line.partition("#")
        if hash_sign and BULLSEYE in comment and "..." in code:
            tags.append(comment.replace(BULLSEYE, "").strip().split(")")[0])
    return tags


def _call(n, name, function, *args, hint=""):
    tags = _blanks(function)
    if tags:
        _say(n, False, "", "%s still has %s on the lines marked %s %s. Replace each ... with your code, run the cell of the piece again, and then run this check again."
             % (name, "a blank" if len(tags) == 1 else "blanks", BULLSEYE, ", ".join(tags)))
        return False, None
    try:
        return True, function(*args)
    except Exception as error:
        _say(n, False, "", "%s stopped with %s: %s. Look at the lines marked %s in the cell above. %s"
             % (name, type(error).__name__, error, BULLSEYE, hint))
        return False, None


def _shape(value):
    return tuple(value.shape) if torch.is_tensor(value) else type(value).__name__


def check_play_days(function):
    hint = "Categorical(probs) makes the distribution, .sample() draws from it and .log_prob(roads) scores the draw."
    # Part 1: the shapes and the kinds of numbers
    torch.manual_seed(1)
    w = torch.zeros(2, 7, requires_grad=True)
    ok, days = _call(1, "play_days", function, w, 5, hint=hint)
    if not ok:
        return
    if not isinstance(days, tuple) or not all(hasattr(days, name) for name in ("places", "roads", "logps", "coins")):
        return _say(1, False, "", "play_days has to return the Days record. Keep the last line of the given code.")
    for name in ("places", "roads", "logps", "coins"):
        value = getattr(days, name)
        if not torch.is_tensor(value) or tuple(value.shape) != (5, 3):
            return _say(1, False, "", "days.%s should be a tensor of shape (5, 3), one row per day and one column per road. Yours is %s. Is the draw in line 1b one road per day?" % (name, _shape(value)))
    if not bool(((days.roads == 0) | (days.roads == 1)).all()):
        return _say(1, False, "", "a road is 0 or 1, and yours are %s. Line 1b draws them from the distribution with .sample()." % days.roads[0].tolist())
    # Part 2: the log-probabilities are those of the roads that were drawn, in the table that played them
    torch.manual_seed(5)
    table = (torch.randn(2, 7) * 1.5).requires_grad_()
    ok, days = _call(1, "play_days", function, table, 300, hint=hint)
    if not ok:
        return
    all_logs = torch.log_softmax(table[:, days.places], dim=0).detach()
    taken = all_logs.gather(0, days.roads.unsqueeze(0)).squeeze(0)
    other = all_logs.gather(0, (1 - days.roads).unsqueeze(0)).squeeze(0)
    got = days.logps.detach()
    if not torch.allclose(got, taken, atol=1e-4):
        if torch.allclose(got, taken.exp(), atol=1e-4):
            return _say(1, False, "", "your numbers are the chances of the roads. A log-probability is the logarithm of a chance: dist.log_prob(roads) gives it directly.")
        if torch.allclose(got, other, atol=1e-4):
            return _say(1, False, "", "your numbers are the log-probabilities of the other road. Line 1c scores the roads that were drawn.")
        k = int(torch.argmax((got - taken).abs().max(dim=1).values))
        return _say(1, False, "", "in day %d the roads %s at the places %s should have the log-probabilities %s and yours are %s." %
                    (k + 1, days.roads[k].tolist(), days.places[k].tolist(), np.round(taken[k].tolist(), 3).tolist(), np.round(got[k].tolist(), 3).tolist()))
    # Part 3: the roads follow the chances of the table. In Eagle Pass road 0 has chance 0.9, in Mirror Lake it has chance 0.1.
    skew = torch.zeros(2, 7)
    skew[0, 0] = float(np.log(9.0))
    skew[1, 1] = float(np.log(9.0))
    skew.requires_grad_()
    torch.manual_seed(7)
    ok, days = _call(1, "play_days", function, skew, 4000, hint=hint)
    if not ok:
        return
    first = days.roads[:, 0]
    share = float((first == 0).float().mean())
    if abs(share - 0.9) > 0.03:
        if share > 0.995:
            return _say(1, False, "", "Mara took road 0 on every day, the road with the larger chance. Draw the road at random from the chances: dist.sample() does that, and taking the largest chance every time does not.")
        if abs(share - 0.5) < 0.05:
            return _say(1, False, "", "the first road looks like a fair coin toss. In Eagle Pass this table gives road 0 the chance 0.9, and road 0 was taken on %.0f%% of 4000 days." % (100 * share))
        return _say(1, False, "", "in Eagle Pass this table gives road 0 the chance 0.9, and road 0 was taken on %.0f%% of 4000 days. Are the chances the ones of the place where each day stands?" % (100 * share))
    second = days.roads[:, 1][first == 0]
    share_second = float((second == 0).float().mean())
    if abs(share_second - 0.1) > 0.04:
        return _say(1, False, "", "after road 0 the day stands in Mirror Lake, where this table gives road 0 the chance 0.1. Road 0 was taken there on %.0f%% of the days. Each day draws its road from the column of the place where it stands." % (100 * share_second))
    # Part 4: PyTorch can follow the log-probabilities back to the knobs
    w2 = torch.zeros(2, 7, requires_grad=True)
    ok, days = _call(1, "play_days", function, w2, 20, hint=hint)
    if not ok:
        return
    if not days.logps.requires_grad:
        return _say(1, False, "", "the log-probabilities carry no gradient, so PyTorch cannot learn from them. Use dist.log_prob(roads) as it is: do not detach it and do not turn it into a Python number.")
    days.logps.sum().backward()
    if w2.grad is None or float(w2.grad.abs().sum()) == 0.0:
        return _say(1, False, "", "the gradient of the log-probabilities does not reach w. Build the distribution from probs, the chances worked out from w in the given line above it.")
    _say(1, True, "A batch of 5 days has shape (5, 3) in every field. The log-probabilities match the table. In Eagle Pass a table with chance 0.9 for road 0 sent %.0f%% of 4000 days along road 0, and in Mirror Lake a table with chance 0.1 sent %.0f%% of the days along road 0. The log-probabilities are connected to w." % (100 * share, 100 * share_second))


def check_pseudo_loss(function):
    torch.manual_seed(2)
    w = (torch.randn(2, 7) * 0.8).requires_grad_()
    days = _fake_days(w)
    multiplier = days.coins.sum(dim=1, keepdim=True).expand(-1, 3)
    hint = "days.logps and multiplier both have shape (4, 3) here. .sum(dim=1) adds up along each row, .mean() averages."
    ok, loss = _call(2, "pseudo_loss", function, days, multiplier, hint=hint)
    if not ok:
        return
    if not torch.is_tensor(loss) or loss.dim() != 0:
        return _say(2, False, "", "pseudo_loss has to return one number (a tensor with a single value), and yours is %s. After 2b there is one number per day, and 2c averages them." % (_shape(loss),))
    if not loss.requires_grad:
        return _say(2, False, "", "the loss is not connected to w. Use days.logps as they are: do not detach them and do not turn them into Python numbers.")
    want = float(_reference_loss(days, multiplier))
    got = float(loss)
    if abs(got - want) > 1e-3 * max(1.0, abs(want)):
        ratio = got / want if want != 0 else float("nan")
        if abs(ratio + 1.0) < 1e-3:
            hint = "The size is right and the sign is wrong: put the minus sign in line 2c. PyTorch minimises the loss, and we want the average coins to go up."
        elif abs(ratio - 1.0 / 3.0) < 1e-3:
            hint = "Your loss is three times too small. The three roads of a day have to be added up (sum) in line 2b. Their average divides by 3."
        elif abs(ratio - 4.0) < 1e-3:
            hint = "Your loss is 4 times too big, and this batch has 4 days. Line 2c averages over the days (mean). A sum over the days grows with the size of the batch."
        elif abs(ratio - 12.0) < 1e-3:
            hint = "Your loss adds up all 12 numbers. After adding up the three roads of each day (line 2b), average over the days in line 2c."
        elif abs(ratio + 4.0) < 1e-3:
            hint = "Your loss has the wrong sign and adds up the days. Line 2c averages over the days (mean) and puts a minus sign in front."
        else:
            hint = "Check the three lines: the product of the log-probability and its multiplier, the sum over the three roads of a day, and the average over the days with a minus sign."
        return _say(2, False, "", "for four days written out by hand the pseudo-loss should be %.2f and yours is %.2f. %s" % (want, got, hint))
    loss.backward()
    mine = -w.grad.detach()
    formula = torch.tensor(hand_estimate(w, days.places, days.roads, multiplier), dtype=torch.float32)
    if not torch.allclose(mine, formula, atol=1e-3, rtol=1e-3):
        k = int(torch.argmax((mine - formula).abs().max(dim=0).values))
        return _say(2, False, "", "the value is right, and the gradient is not the formula's. In the column of %s autograd gives %s and the formula gives %s." % (PLACES[k], np.round(mine[:, k].tolist(), 3).tolist(), np.round(formula[:, k].tolist(), 3).tolist()))
    _say(2, True, "For four days written out by hand the pseudo-loss is %.2f. Minus its gradient is the formula (1/B) * sum of multiplier * score on all 14 knobs: for the knob of road 0 in Eagle Pass autograd gives %.3f and the formula gives %.3f."
         % (got, mine[0, 0], formula[0, 0]))


def check_reinforce_step(function):
    torch.manual_seed(3)
    lr = 0.05
    w = (torch.randn(2, 7) * 0.8).requires_grad_()
    optimiser = torch.optim.SGD([w], lr=lr)
    # count the calls of the optimiser's step, so that the message can say which of the three lines is missing
    counts = {"step": 0, "backward": 0}
    real_step = optimiser.step

    def counted_step(*args, **kwargs):
        counts["step"] += 1
        return real_step(*args, **kwargs)

    optimiser.step = counted_step
    hint = "The loss is already computed, and optimiser, loss are the two arguments."
    previous = None
    for number in (1, 2):
        # a batch of days written by hand, with log-probabilities worked out from the table as it is now
        days = _fake_days(w)
        multiplier = days.coins.sum(dim=1, keepdim=True).expand(-1, 3)
        loss = _reference_loss(days, multiplier)
        loss.register_hook(lambda gradient: counts.update(backward=counts["backward"] + 1))
        counts["step"], counts["backward"] = 0, 0
        before = w.detach().clone()
        hand = torch.tensor(hand_estimate(before, days.places, days.roads, multiplier), dtype=torch.float32)
        ok, _ = _call(3, "reinforce_step", function, optimiser, loss, hint=hint)
        if not ok:
            return
        moved = w.detach() - before
        want = lr * hand
        if torch.allclose(moved, want, atol=1e-4, rtol=1e-3):
            previous = hand
            continue
        if float(moved.abs().max()) == 0.0:
            if counts["backward"] == 0:
                return _say(3, False, "", "w did not move, and loss.backward() was never called, so there is no gradient to step with (line 3b).")
            if counts["step"] == 0:
                return _say(3, False, "", "w did not move, and optimiser.step() was never called (line 3c).")
            return _say(3, False, "", "w did not move, although backward and step were both called. The three lines have to come in this order: forget the old gradients (3a), compute the new gradient (3b), take the step (3c).")
        if torch.allclose(moved, -want, atol=1e-4, rtol=1e-3):
            return _say(3, False, "", "w moved, but downhill on the average coins. The step of the optimiser goes downhill on the loss, and the loss has a minus sign in front, so the average coins should go up. Check that the lines are the three standard ones.")
        if number == 2 and previous is not None and torch.allclose(moved, lr * (hand + previous), atol=1e-4, rtol=1e-3):
            return _say(3, False, "", "in the second update w moved by the new gradient plus the one of the first update. PyTorch adds a new gradient to the old one, so every update must start with optimiser.zero_grad() (line 3a).")
        return _say(3, False, "", "after update %d w is not where the formula puts it (w + lr * g). The biggest difference is %.4f. Check the three lines and their order." % (number, float((moved - want).abs().max())))
    _say(3, True, "Two updates in a row on batches written by hand moved w by exactly lr * g each time, with g from the formula. Nothing is left over from one update to the next.")


def check_reward_to_go(function):
    coins = torch.tensor([[7.0, 7.0, 7.0], [9.0, 4.0, 2.0], [0.0, 5.0, 0.0], [3.0, 1.0, 4.0]])
    want = torch.tensor([[21.0, 14.0, 7.0], [15.0, 6.0, 2.0], [5.0, 5.0, 0.0], [8.0, 5.0, 4.0]])
    hint = "coins has shape (4, 3) here. flip(1) reverses every row and cumsum(dim=1) adds up along every row."
    ok, got = _call(4, "reward_to_go", function, coins, hint=hint)
    if not ok:
        return
    if not torch.is_tensor(got) or tuple(got.shape) != (4, 3):
        return _say(4, False, "", "reward_to_go has to return a tensor of shape (4, 3) here, one number for every road of every day, and yours is %s." % (_shape(got),))
    if not torch.allclose(got.float(), want):
        if torch.allclose(got.float(), coins.cumsum(dim=1)):
            return _say(4, False, "", "these are the coins collected so far. The coins to go are the ones from this road to the end of the day: for 9, 4, 2 they are 15, 6, 2. Read the day backwards before adding up (4a), and flip the result back (4b).")
        if torch.allclose(got.float(), want.flip(1)):
            return _say(4, False, "", "the numbers are the right ones, in the reverse order. Flip the running totals back into the order of the day in line 4b.")
        if torch.allclose(got.float(), coins.sum(dim=1, keepdim=True).expand(-1, 3)):
            return _say(4, False, "", "these are the coins of the whole day at every road, the multiplier of before. The coins to go shrink along the day: for 9, 4, 2 they are 15, 6, 2.")
        k = int(torch.argmax((got.float() - want).abs().max(dim=1).values))
        return _say(4, False, "", "the coins %s should give the coins to go %s and yours give %s." % (coins[k].tolist(), want[k].tolist(), np.round(got[k].tolist(), 3).tolist()))
    torch.manual_seed(4)
    random_coins = torch.randint(0, 10, (50, 3)).float()
    expected = torch.stack([random_coins[:, 0] + random_coins[:, 1] + random_coins[:, 2], random_coins[:, 1] + random_coins[:, 2], random_coins[:, 2]], dim=1)
    if not torch.allclose(function(random_coins).float(), expected):
        return _say(4, False, "", "on 50 random days some of the coins to go are wrong. Each day is its own row: flip and cumsum along dim 1.")
    _say(4, True, "A day that paid 7, 7, 7 gives the coins to go 21, 14, 7, and a day that paid 9, 4, 2 gives 15, 6, 2. The first road is credited with the whole day, the last road with its own arrival only.")


def pieces_ready(*numbers):
    missing = [str(n) for n in numbers if not _passed.get(n)]
    if missing:
        print("Not yet. %s %s %s not passed %s check. Finish %s, run %s check cell, and then run this cell again."
              % ("Piece" if len(missing) == 1 else "Pieces", ", ".join(missing), "has" if len(missing) == 1 else "have",
                 "its" if len(missing) == 1 else "their", "it" if len(missing) == 1 else "them", "its" if len(missing) == 1 else "their"))
        return False
    return True


# ======================================================================
# Looks that use your pieces
# ======================================================================

def show_one_day():
    # one day written by hand: Eagle Pass, Mirror Lake, Vine Terraces, Olive Grove Estate, 7 + 7 + 7 coins, starting table
    w = torch.zeros(2, 7, requires_grad=True)
    places = torch.tensor([[0, 1, 4]])
    roads = torch.tensor([[0, 1, 0]])
    coins = torch.tensor([[7.0, 7.0, 7.0]])
    logps = torch.log_softmax(w[:, places], dim=0).gather(0, roads.unsqueeze(0)).squeeze(0)
    days = Days(places, roads, logps, coins)
    multiplier = coins.sum(dim=1, keepdim=True).expand(-1, 3)
    pseudo_loss(days, multiplier).backward()
    from_loss = -w.grad.detach() + 0.0                  # + 0.0 turns a negative zero into a plain zero
    formula = torch.tensor(hand_estimate(w, places, roads, multiplier), dtype=torch.float32) + 0.0
    print("The day: Eagle Pass (road 0) -> Mirror Lake (road 1) -> Vine Terraces (road 0) -> Olive Grove Estate, 7 + 7 + 7 = 21 coins, every knob at 0.")
    print()
    print("%-16s %-11s %-7s %-20s %s" % ("", "", "", "push from your loss", "push from the formula"))
    print("%-16s %-11s %-7s %9s %9s    %-26s %s" % ("place", "road taken", "chance", "knob 0", "knob 1", "knob 0", "knob 1"))
    taken = {0: 0, 1: 1, 4: 0}
    for i in range(7):
        texts = ["", ""]
        chosen, chance = "-", "-"
        if i in taken:
            chosen, chance = "road %d" % taken[i], "0.50"
            for knob in (0, 1):
                factor = "(1 - 0.50)" if knob == taken[i] else "(-0.50)"
                texts[knob] = "21 x %s = %+.2f" % (factor, formula[knob, i])
        print("%-16s %-11s %-7s %+9.2f %+9.2f    %-26s %s" % (PLACES[i], chosen, chance, from_loss[0, i], from_loss[1, i], texts[0], texts[1]))


def show_average():
    # 300 batches of 100 days at the starting table: the average of the estimates against the exact gradient
    exact = exact_gradient(np.zeros((2, 7)))[0]
    estimates = []
    for k in range(300):
        torch.manual_seed(1000 + k)
        w = torch.zeros(2, 7, requires_grad=True)
        days = play_days(w, 100)
        multiplier = days.coins.sum(dim=1, keepdim=True).expand(-1, 3)
        pseudo_loss(days, multiplier).backward()
        estimates.append((-w.grad.detach())[0])
    estimates = torch.stack(estimates)
    average = estimates.mean(dim=0)
    print("the knob of road 0 at each place    exact gradient    average of 300 estimates    one single estimate    spread of the 300")
    for i in range(7):
        print("%-35s %+10.3f %+18.3f %+24.3f %18.3f" % (PLACES[i], exact[i], average[i], estimates[0][i], estimates[:, i].std()))


# ======================================================================
# The pictures
# ======================================================================

INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SURFACE, BLUE, RUST = "#fcfcfb", "#2a78d6", "#c4501b"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#256abf", "#0d366b"])


def _style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)


def _days_axis(ax, logx, first, last):
    # the days played on the horizontal axis, with thousands separators; a logarithmic axis gets round ticks
    if logx:
        ax.set_xscale("log")
        ax.set_xticks([t for t in (100, 200, 500, 1000, 2000, 5000, 10000, 20000) if first * 0.99 <= t <= last * 1.01])
        ax.xaxis.set_minor_formatter(NullFormatter())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, position: "{:,.0f}".format(x)))


def draw_road(ax, w):
    # the fifteen places as a tree, the chance of every road written on it, the best path outlined
    chances = first_road_chances(w)
    position = {}
    for i in range(15):
        level = int(np.floor(np.log2(i + 1)))
        k = i - (2 ** level - 1)
        position[i] = ((k + 0.5) * 8.0 / 2 ** level, 3.0 - level)
    best_places = [0] + places_of(best_path())
    best_edges = set(zip(best_places[:-1], best_places[1:]))
    ax.set_facecolor(SURFACE)
    for i in range(7):
        for road in (0, 1):
            child = 2 * i + 1 + road
            chance = chances[i] if road == 0 else 1.0 - chances[i]
            (x0, y0), (x1, y1) = position[i], position[child]
            if (i, child) in best_edges:
                ax.plot([x0, x1], [y0, y1], color=INK, linewidth=2 + 6 * chance + 2.5, solid_capstyle="round", zorder=1)
            ax.plot([x0, x1], [y0, y1], color=BLUES(0.15 + 0.85 * chance), linewidth=2 + 6 * chance, solid_capstyle="round", zorder=2)
            side = -1 if road == 0 else 1
            ax.text(x0 + 0.68 * (x1 - x0) + side * 0.17, y0 + 0.68 * (y1 - y0), "%.0f%%" % (100 * chance), ha="center", va="center",
                    fontsize=9.5, color=INK, zorder=4, bbox=dict(boxstyle="round,pad=0.12", facecolor=SURFACE, edgecolor="none", alpha=0.9))
    for i, (x, y) in position.items():
        ax.scatter([x], [y], s=330, color=SURFACE, edgecolor=INK if i in best_places else MUTED, linewidth=2.2 if i in best_places else 1, zorder=3)
        ax.text(x, y, str(i), ha="center", va="center", fontsize=9, color=INK if i in best_places else SECONDARY, zorder=5)
    ax.set_xlim(-0.2, 8.2)
    ax.set_ylim(-0.45, 3.4)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_title("The table of chances on the road (seed 0)", color=INK, fontsize=12, loc="left")
    ax.set_xlabel("outlined: the best path. Each road shows its chance.", color=SECONDARY, fontsize=9.5)


def summarise_runs(runs, B):
    best = best_path()
    found = sum(1 for w, curve in runs if most_likely_path(w)[0] == best)
    finals = [curve[-1] for w, curve in runs]
    reached = [days_to_share(curve, B) for w, curve in runs]
    reached = [d for d in reached if d is not None]
    return found, finals, reached


def show_runs(runs, B, base=None, logx=False):
    # left: the exact average coins of every run against the days played; right: the road of the first run
    start, best = runs[0][1][0], float(path_coins().max())
    fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1.1, 1]}, facecolor=SURFACE)
    _style_axes(left)
    if base is not None:
        for k, (w, curve) in enumerate(base):
            left.plot(B * np.arange(1, len(curve)), curve[1:], color=MUTED, linewidth=1.0, alpha=0.6, label="before the switches" if k == 0 else None)
    for k, (w, curve) in enumerate(runs):
        first = 1 if logx else 0
        left.plot(B * np.arange(first, len(curve)), curve[first:], color=BLUE, linewidth=2.2 if k == 0 else 1.3, alpha=1.0 if k == 0 else 0.6,
                  label=("this run's switches" if base is not None else "one run per seed") if k == 0 else None)
    left.axhline(start, color=MUTED, linewidth=1)
    left.axhline(best, color=MUTED, linewidth=1)
    last = B * (len(runs[0][1]) - 1)
    _days_axis(left, logx, B if logx else 0, last)
    left_edge = B if logx else 0
    left.text(left_edge, start - 0.1, "start, every knob at 0: %.2f" % start, ha="left", va="top", color=SECONDARY, fontsize=10)
    left.text(left_edge, best + 0.1, "the best path pays %d on average" % best, ha="left", va="bottom", color=SECONDARY, fontsize=10)
    left.set_ylim(start - 0.9, best + 0.8)
    left.set_xlabel("days played" + (" (logarithmic axis)" if logx else ""), color=SECONDARY)
    left.set_ylabel("average coins of a day, exact", color=SECONDARY)
    left.set_title("Mara's average coins as REINFORCE learns", color=INK, fontsize=12, loc="left")
    left.legend(loc="lower right", frameon=False, labelcolor=SECONDARY, fontsize=9.5)
    draw_road(right, runs[0][0])
    fig.tight_layout()
    plt.show()
    found, finals, reached = summarise_runs(runs, B)
    print("%d runs, %s days played in each." % (len(runs), "{:,}".format(B * (len(runs[0][1]) - 1))))
    print("The best path is the most likely path in %d of %d runs." % (found, len(runs)))
    print("Exact average coins at the end: median %.2f, lowest %.2f. The start was %.2f and the best path pays %d." % (np.median(finals), min(finals), start, best))
    if reached:
        print("The runs that closed 90%% of the gap between the start and the best path did it after %s days (median of %d runs)." % ("{:,.0f}".format(np.median(reached)), len(reached)))
    else:
        print("No run closed 90% of the gap between the start and the best path.")


def show_lock_in(runs, B):
    # the curves of the runs of a large step, coloured by the path each run ends on
    best = best_path()
    start, best_pay = runs[0][1][0], float(path_coins().max())
    fig, ax = plt.subplots(figsize=(8.6, 4.4), facecolor=SURFACE)
    _style_axes(ax)
    for k, (w, curve) in enumerate(runs):
        good = most_likely_path(w)[0] == best
        ax.plot(B * np.arange(1, len(curve)), curve[1:], color=BLUE if good else RUST, linewidth=1.8, alpha=0.85)
    ax.axhline(start, color=MUTED, linewidth=1)
    ax.axhline(best_pay, color=MUTED, linewidth=1)
    ax.set_ylim(start - 0.5, best_pay + 0.6)
    _days_axis(ax, True, B, B * (len(runs[0][1]) - 1))
    ax.set_xlabel("days played (logarithmic axis)", color=SECONDARY)
    ax.set_ylabel("average coins of a day, exact", color=SECONDARY)
    ax.set_title("Blue: ends on the best path. Orange: ends on another path", color=INK, fontsize=12, loc="left")
    fig.tight_layout()
    plt.show()
    coins = path_coins()
    for k, (w, curve) in enumerate(runs):
        path, probability = most_likely_path(w)
        if path != best:
            print("seed %d ends on a path that pays %d coins on average, which it takes on %.0f%% of its days" % (k, coins[ALL_PATHS.index(path)], 100 * probability))
    print("%d of %d runs end on the best path." % (sum(1 for w, c in runs if most_likely_path(w)[0] == best), len(runs)))


print("Ready. PyTorch %s is loaded. Nothing was installed and nothing was downloaded." % torch.__version__)

## The road

Nothing to fill in here. It is the road of the first exercise: fifteen places numbered row by row from Eagle Pass, which is place 0. The two roads out of place `i` lead to place `2i + 1` (road 0) and place `2i + 2` (road 1). Places 0 to 6 have a choice. Places 7 to 14 are where a day ends. A path is the way of three roads from Eagle Pass to one of those eight places, and every day follows one path.

`COINS[i]` is the usual pay of place `i`, what it pays Mara on average. Every arrival pays the usual coins plus a luck of `-LUCK` to `+LUCK` coins, all equally likely and drawn anew at every arrival. With `LUCK = 1` the luck is -1, 0 or +1.

`world_step(places, roads)` is the world's answer to a road. It takes the places where some days stand and the roads they take, and returns the places they arrive at and the coins they get. It plays the part of `env.step(action)` in the popular codes. REINFORCE sees these coins and never looks at `COINS` or at the luck.

`Days` is the record of a batch of days that your `play_days` will return: the place where each road was chosen, the road, its log-probability and the coins of the arrival. Every field has one row per day and one column per road, so the shape is `(B, 3)`.

`exact_J(w)` is given too. It adds up the eight paths, each weighted by its chance under the table `w`, and returns the exact average coins of a day. It draws the plots and runs the checks. The method never calls it.

In [ ]:
# The fifteen places in the order of their numbers, and what each one pays on average when Mara arrives.
PLACES = ["Eagle Pass", "Mirror Lake", "Windmill Market", "Millbrook", "Vine Terraces",
          "Red Canyon", "Dune Inn", "Fishing Harbour", "Reef Lighthouse",
          "Olive Grove Estate", "Citrus Market", "Cliff Monastery", "Hot Springs",
          "Salt Oasis", "Smugglers' Cove"]
COINS = np.array([0, 7, 9, 8, 7, 3, 4, 4, 3, 7, 4, 2, 7, 2, 1])
COINS_T = torch.tensor(COINS)

# The luck of an arrival goes from -LUCK to +LUCK coins. The smallest usual pay is 1,
# so LUCK = 1 is the largest size that never pays a negative amount.
LUCK = 1


def world_step(places, roads):
    # places and roads hold one number per day. The world answers with the places the days arrive at and their coins.
    next_places = 2 * places + 1 + roads                    # the roads out of place i lead to 2i + 1 and 2i + 2
    luck = torch.randint(-LUCK, LUCK + 1, places.shape)     # -1, 0 or +1, equally likely, drawn anew for every day
    coins = (COINS_T[next_places] + luck).float()           # the usual coins of the place just reached, plus the luck
    return next_places, coins


# What play_days returns for a batch of B days: four tensors of shape (B, 3), one row per day and one column per road.
Days = namedtuple("Days", ["places", "roads", "logps", "coins"])

show_road()
print()
print("The eight possible paths pay", path_coins().tolist(), "coins on average.")
print("With every knob at 0, Mara tosses a fair coin at every fork. The exact average of her day is", exact_J(torch.zeros(2, 7)), "coins.")
print()
show_path([1, 4, 9])    # one path: the three places Mara visits after Eagle Pass

## Where this code comes from

Two programs are the usual starting points when somebody learns REINFORCE in PyTorch.

- **The PyTorch example**, `reinforcement_learning/reinforce.py` in the PyTorch examples repository. It trains CartPole, one episode per update.
  https://github.com/pytorch/examples/blob/main/reinforcement_learning/reinforce.py
- **Spinning Up**, OpenAI's teaching resource by Joshua Achiam (2018). Part 3, "Intro to Policy Optimization", derives the simplest policy gradient and codes it in `1_simple_pg.py`, with one batch of episodes per update.
  https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html
  https://github.com/openai/spinningup/blob/master/spinup/examples/pytorch/pg_math/1_simple_pg.py

The Hugging Face Deep RL course follows the PyTorch example in its Unit 4:
https://huggingface.co/learn/deep-rl-course/unit4/hands-on

The algorithm itself is from R. J. Williams (1992), "Simple statistical gradient-following algorithms for connectionist reinforcement learning", *Machine Learning* 8, pages 229 to 256.

This notebook writes its own code for Mara's road. The table shows where each part of the popular code lives in it.

| the popular code has | how it looks there | in this notebook |
|--|--|--|
| a policy that gives chances | `Policy`, a small network ending in a softmax (PyTorch example); `logits_net` and `get_policy` (Spinning Up) | the table `w` and `torch.softmax` |
| a draw and its log-probability | `Categorical(probs)`, `m.sample()` and `m.log_prob(action)`, kept in a list (PyTorch example); `get_policy(obs).log_prob(act)` (Spinning Up) | Piece 1 |
| a multiplier for every step | the discounted `returns`, with gamma 0.99 (PyTorch example); the return of the whole episode for every step (Spinning Up) | the coins of the whole day, then the coins to go (Piece 4) |
| standardising | `(returns - returns.mean()) / (returns.std() + eps)` (PyTorch example) | the switch `STANDARDISE` |
| the pseudo-loss | `-log_prob * R`, summed (PyTorch example); `-(logp * weights).mean()` (Spinning Up) | Piece 2 |
| the update | `optimizer.zero_grad()`, `backward()` and `step()` in both | Piece 3 |
| the optimiser | Adam with `lr=1e-2` in both | plain SGD, with the switch `ADAM` |
| the days of one update | one episode (PyTorch example); a batch of episodes (Spinning Up) | a batch of `B` days |

## A minute with tensors

Nothing to fill in. PyTorch keeps its numbers in **tensors**, which are tables of numbers. In this notebook a tensor usually has one row for every day and one column for every road. Run the cell below and read what it prints. It shows four things about tensors that the pieces need.

1. **Shape.** `coins.shape` gives the number of rows and the number of columns. Two days of three roads each have the shape `(2, 3)`.
2. **Entry by entry.** `coins * 10` multiplies every entry by 10. In the same way `a * b` multiplies two tensors of the same shape entry by entry.
3. **`dim`.** Many methods take `dim=`, which says in which direction to work. `dim=1` means along each row, so every row (here: every day) is handled on its own and the result has one number per row. `dim=0` works down each column instead, and gives one number per column.
4. **One number from many.** `coins.sum(dim=1)` adds up the numbers of each row. `coins.mean()` averages all the numbers of a tensor into one number.

In [ ]:
# Two days (the rows) of three roads each (the columns): the coins that every arrival paid.
coins = torch.tensor([[7., 7., 7.],
                      [9., 4., 2.]])
print("coins has shape", tuple(coins.shape), "(2 rows, 3 columns)")
print()
print("coins * 10 multiplies every entry by 10:")
print(coins * 10)
print()
print("coins.sum(dim=1) adds up each row, one number per day:", coins.sum(dim=1))
print("coins.sum(dim=0) adds up each column, one number per road:", coins.sum(dim=0))
print("coins.mean() averages all six numbers:", coins.mean())

## Piece 1: the policy plays days

Mara's table of knobs is a tensor `w` with 2 rows (one for every road) and 7 columns (one for every place that has a fork), as in the slides. We create it with `requires_grad=True`, which tells PyTorch to remember every computation that involves `w`, so that it can compute gradients later. Every knob starts at 0.

`play_days(w, B)` plays `B` days at once, so every tensor in it has one row for every day. All days start in Eagle Pass and take three roads. At every fork the code does four things for all the days together:

1. read the two knobs of the place where each day stands, turn them into the chances of its two roads with a softmax, which makes two numbers that add up to 1, and lay the chances of every day in a row (a given line, explained below the table);
2. make a **distribution** from those chances (line 1a). A distribution is a recipe for random draws: it knows how likely each road is, and it can draw roads at random with exactly those chances;
3. **draw** one road for every day from the distribution (line 1b);
4. ask the distribution for the **log-probability** of every road that was drawn (line 1c). It is the logarithm of the chance of that road. A road with chance 0.5 has the log-probability -0.69, a road with chance 0.9 has -0.11, and a rare road with chance 0.1 has -2.30. PyTorch remembers how every log-probability depends on `w`, and the next piece uses exactly that.

### The names in this piece

| name | what it holds | shape |
|--|--|--|
| `places` | the place where each day stands, one whole number per day (every day starts in Eagle Pass, place 0) | `(B,)` |
| `w[:, places]` | the two knobs of the place where each day stands: column k is for day k | `(2, B)` |
| `probs` | the chances of the two roads for every day: row k is `[chance of road 0, chance of road 1]` for day k, and every row adds up to 1. The given line makes it with `torch.softmax(w[:, places], dim=0).T`: the softmax handles each column on its own, and `.T` lays the columns down as rows | `(B, 2)` |
| `dist` | the distribution made from `probs`. The name is ours: you create it in line 1a | an object that holds `probs` |
| `roads` | the roads that were drawn, a 0 or a 1 for every day | `(B,)` |
| `logps` | the log-probability of the road that each day took, one number per day | `(B,)` |

### Reading the table

The given line of `play_days` picks the knobs of the places out of the table and turns them into chances. It uses three tools. The cell below shows them on a toy table of 2 rows (the roads) and 3 columns (the places), with three days that stand at the places 2, 0 and 2. Run it and read what it prints.

- `toy_table[:, toy_places]` picks columns. The colon means all the rows, and `toy_places` lists the columns to take, one for each day, so the result has one column for every day.
- `torch.softmax(x, dim=0)` turns the numbers of every column of `x` into chances that add up to 1. `dim=0` is the direction down each column, as in the minute with tensors.
- `x.T` swaps rows and columns, so that the chances of every day lie in a row, which is the shape that `Categorical` reads in the next cell.

In [ ]:
# A toy table of knobs: 2 rows (road 0 and road 1) and 3 columns (three places).
toy_table = torch.tensor([[0.0, 1.0, 2.0],
                          [0.0, 0.0, 0.0]])
toy_places = torch.tensor([2, 0, 2])                  # three days stand at the places 2, 0 and 2

toy_knobs = toy_table[:, toy_places]                  # the column of the place of every day: shape (2, 3)
toy_chances = torch.softmax(toy_knobs, dim=0)         # every column becomes two chances that add up to 1
print("the knobs of the three days, one column for each day:")
print(toy_knobs)
print()
print("their chances, still one column for each day (every column adds up to 1):")
print(toy_chances)
print()
print("toy_chances.T has one row for each day, so its shape is", tuple(toy_chances.T.shape), ":")
print(toy_chances.T)

### The tools

The cell below uses the three tools on three days that stand at three different places, with their chances in `probs`: the class `Categorical` (already imported for you) and the two methods `.sample()` and `.log_prob(...)` of the distribution that it makes. Run it and read what it prints. Run it again to see that the roads change from run to run and the chances do not.

In [ ]:
# Three days stand at three different places. Row k holds the chances of road 0 and road 1 for day k.
probs = torch.tensor([[0.5, 0.5],
                      [0.9, 0.1],
                      [0.2, 0.8]])

dist = Categorical(probs)        # a distribution made from the chances: it knows one row of chances for every day
roads = dist.sample()            # draw one road for every day: a tensor with a 0 or a 1 per day
logps = dist.log_prob(roads)     # the log-probability of every road that was drawn: one number per day

show_draws(probs, roads, logps)  # prints one line per day (given)

**What to write.** In `play_days` the same three tools work on the real `probs`, which has one row for every day, as in the cell above:

- **1a)** `dist`: a distribution made from `probs`, with `Categorical(...)`;
- **1b)** `roads`: a draw from `dist`, with its method `.sample()`;
- **1c)** `logps`: the log-probabilities of the roads that were drawn, asked from `dist` with its method `.log_prob(...)`, which takes the drawn `roads` as its argument.

🎯 **Piece 1.** Complete the three lines marked 1a, 1b and 1c.

<details><summary>💡 Hint</summary>

Do what the cell above did, with the names of the function: `Categorical(` the chances `)`, then `.sample()` called on the distribution, then `.log_prob(` the roads `)` called on the distribution.
</details>

In [ ]:
def play_days(w, B):
    places = torch.zeros(B, dtype=torch.long)          # every day starts in Eagle Pass, place 0
    saved = {"places": [], "roads": [], "logps": [], "coins": []}
    for step in range(3):                              # a day is three roads long
        probs = torch.softmax(w[:, places], dim=0).T   # the chances of the two roads where each day stands: shape (B, 2)
        dist = ...                                     # 🎯 1a) a distribution over the two roads, one row of chances per day
        roads = ...                                    # 🎯 1b) draw one road for every day: a 0 or a 1
        logps = ...                                    # 🎯 1c) the log-probability of the roads that were drawn
        saved["places"].append(places)
        saved["roads"].append(roads)
        saved["logps"].append(logps)
        places, coins = world_step(places, roads)      # the world answers with the next places and the coins of the arrival
        saved["coins"].append(coins)
    return Days(**{name: torch.stack(items, dim=1) for name, items in saved.items()})    # every field has shape (B, 3)

In [ ]:
check_play_days(play_days)

### Look: three days at the starting table

With every knob at 0 each road has chance 0.5, so a day has chance 0.5 × 0.5 × 0.5 = 0.125. The chances of the three roads multiply, and the logarithm of a product is the sum of the logarithms. So the three log-probabilities of a day add up to the log-probability of its three roads together, log 0.125 = -2.08. The cell below shows three days played with your `play_days`.

In [ ]:
if pieces_ready(1):
    torch.manual_seed(1)
    w = torch.zeros(2, 7, requires_grad=True)     # the table of knobs: 2 rows (the roads), 7 columns (the places), every knob at 0
    show_days(play_days(w, 3))

## Piece 2: the pseudo-loss

Now the heart of REINFORCE. For every road that Mara took, we want to push the knobs so that this road becomes more likely, and push harder when its day paid more.

"Make this road more likely" means the gradient of its log-probability `logp` with respect to the knobs. We call this gradient the **score** of the road. For a column of two knobs it has a simple shape: the knob of the road that was taken moves by 1 minus the chance of that road, and the other knob moves by minus that chance. A road with chance 0.5 has the score +0.5 and -0.5. A rare road with chance 0.1 has the score +0.9 and -0.9: surprising roads get big scores.

The number that multiplies a score is its **multiplier**. For now the multiplier of all three roads of a day is `G`, the coins of the whole day. The estimate of the gradient of the average coins is the average over the days:

$$g = \frac{1}{B}\sum_{\text{days}}\ \sum_{\text{roads}}\ \text{multiplier}\times\text{score}$$

PyTorch computes gradients of losses, so we build a loss whose gradient is minus `g`:

$$\text{loss} = -\frac{1}{B}\sum_{\text{days}}\ \sum_{\text{roads}}\ \text{multiplier}\times\text{logp}$$

The multiplier is a plain number that PyTorch treats as a constant, so the gradient of `multiplier * logp` is `multiplier * score`. Stepping downhill on this loss therefore moves the knobs uphill on the average coins. Two notes on it:

- It is called a **pseudo-loss**, because its value means nothing. It does not measure how good the table is. It has the right gradient at the table that played these days, and only there. After one step the table has changed, and we play new days.
- There is a second way to read it. The three log-probabilities of a day add up to the log-probability of the whole day, so with `G` at every road a day contributes `G × logp(day)`. Minimising the loss means maximising the log-likelihood of Mara's own days, in which the richer days count more.

### The names in this piece

`pseudo_loss(days, multiplier)` gets two arguments. This is what they hold, and what the three lines create:

| name | what it holds | shape |
|--|--|--|
| `days` | the record of the batch that `play_days` of Piece 1 returned. It has four fields, `days.places`, `days.roads`, `days.logps` and `days.coins`, and each field is a tensor with one row for every day and one column for every road | each `(B, 3)` |
| `days.logps` | the log-probability of every road that was taken, as Piece 1 saved it. It is the only field of `days` that the pseudo-loss needs | `(B, 3)` |
| `multiplier` | the number that multiplies the score of every road. For now it is the coins of the whole day, repeated for the three roads of that day | `(B, 3)` |
| `weighted` | every log-probability times its multiplier (line 2a) | `(B, 3)` |
| `per_day` | the three weighted numbers of each day added up (line 2b) | `(B,)` |

### The tools

Three small tools build the loss. The cell below uses all of them on two days with made-up numbers. Run it and follow the shapes: `(2, 3)`, then `(2,)`, then a single number.

- `a * b` multiplies two tensors of the same shape entry by entry, so every log-probability meets its own multiplier.
- `x.sum(dim=1)` adds up along each row, here the three roads of each day, and gives one number per day. (`dim=1` is the direction of the minute with tensors: along each row.)
- `x.mean()` averages all the numbers of a tensor into one number. The loss has to be a single number, so after adding up the roads of each day we average over the days.

The minus sign comes last. PyTorch makes a loss small and we want the average coins to go up, so the loss is minus the estimate: making it small makes the coins grow.

In [ ]:
# Two days of three roads: made-up log-probabilities (what days.logps holds) and a multiplier for every road.
logps = torch.tensor([[-0.7, -0.7, -0.7],
                      [-0.1, -2.3, -0.7]])
multiplier = torch.tensor([[21., 21., 21.],
                           [15., 15., 15.]])

product = multiplier * logps       # entry by entry: shape (2, 3)
row_sums = product.sum(dim=1)      # add up each row: one number per day, shape (2,)
average = row_sums.mean()          # average over the two days: a single number
print("multiplier * logps has shape", tuple(product.shape), ":")
print(product)
print("sum(dim=1) has shape", tuple(row_sums.shape), ", one number per day:", row_sums)
print("mean() of those two is a single number:", average, " and minus that:", -average)

**What to write.** The same three steps on the real `days.logps` and `multiplier`, which both have the shape `(B, 3)`:

- **2a)** `weighted`: every log-probability times its multiplier, with `*`;
- **2b)** `per_day`: the three roads of each day added up, with `.sum(dim=1)`;
- **2c)** the value to return: the average over the days, with `.mean()`, and a minus sign in front.

🎯 **Piece 2.** Complete the three lines marked 2a, 2b and 2c.

<details><summary>💡 Hint</summary>

As in the cell above: `multiplier * ` the log-probabilities (they are `days.logps`), then `.sum(dim=1)` called on the result, then `.mean()` called on that, with a minus sign in front.
</details>

In [ ]:
def pseudo_loss(days, multiplier):
    # days.logps and multiplier both have shape (B, 3): one number for every road of every day
    weighted = ...                              # 🎯 2a) every log-probability times its multiplier
    per_day = ...                               # 🎯 2b) add up the three roads of each day: shape (B,)
    return ...                                  # 🎯 2c) average over the days, with a minus sign in front

In [ ]:
check_pseudo_loss(pseudo_loss)

### Look: one day, three pushes

Your pseudo-loss gives PyTorch a single number, and `backward()` turns it into a gradient. Before we trust it, let us look at what it computes for one day. The cell below takes a day that went Eagle Pass, Mirror Lake, Vine Terraces, Olive Grove Estate and paid 7 + 7 + 7 = 21 coins, with the table at the start. For every place it prints the push that your loss gives to the two knobs of the column (minus the gradient of the loss), and the push that the formula gives: `21 × (1 - chance)` for the knob of the road that was taken and `21 × (-chance)` for the other knob.

One day pushes the three columns it walked through and leaves the other four columns alone. A batch of days touches the whole table.

In [ ]:
if pieces_ready(2):
    show_one_day()

### ⭐ Optional: one estimate, and the average of many

A single batch gives a noisy estimate. Is it right on average? The cell below plays 300 batches of 100 days at the starting table. For the knob of road 0 at each place it prints the exact gradient of the average coins (worked out by `exact_J`, nudging the knob up and down), the average of the 300 estimates, one single estimate, and the spread of the 300 estimates.

The average of many estimates lands on the exact gradient. One batch can be far off, so the step size will matter.

In [ ]:
if pieces_ready(1, 2):
    show_average()

## Piece 3: one update

Piece 2 built a **loss**: one number computed from the knobs `w` and a batch of days. A real loss says how bad a table is, and training makes it small. Ours is a pseudo-loss, built so that making it small makes Mara's average coins grow. PyTorch remembers how the number was computed from `w`, so it can work out the gradient of the loss with respect to `w`: how the loss changes when each of the fourteen knobs changes. It is the gradient of the recap, for all the knobs at once.

An **optimiser** is the PyTorch object that moves the knobs. We create it once, with the knobs it has to move and the step size: `torch.optim.SGD([w], lr=alpha)`. The given `train` does this and hands the optimiser to your function together with the loss. `lr` is PyTorch's name for the step size, our `alpha`, and SGD is the grasshopper's rule: with the gradient of the loss in `w.grad`, one step does `w = w - lr * w.grad`, which goes downhill on the loss. The gradient of our loss is minus `g`, so this is `w = w + lr * g`, the grasshopper's `x = x + alpha * g` of the recap, and the average coins climb.

An update uses three methods, in this order:

1. `optimiser.zero_grad()` makes the optimiser forget the gradients of the previous update. PyTorch adds a new gradient to the old one unless it is told to forget;
2. `loss.backward()` asks the loss for its gradient and stores it in `w.grad`;
3. `optimiser.step()` moves the knobs by one step, using the gradient that is stored.

### The tools

The cell below runs the three methods on a toy: one knob, called `knob`, and the loss `(knob - 3) ** 2`, which is smallest at 3. The optimiser holds the knob and the step size 0.1. Run it and read the three updates: each one finds the gradient of the loss and moves the knob towards 3.

In [ ]:
# A toy with one knob. Its loss (knob - 3) ** 2 is smallest at knob = 3.
knob = torch.tensor([0.0], requires_grad=True)         # the knob, starting at 0
toy_optimiser = torch.optim.SGD([knob], lr=0.1)        # the optimiser: it holds the knob and the step size 0.1

for update in range(1, 4):
    toy_loss = (knob - 3) ** 2                         # the loss at the current knob
    toy_optimiser.zero_grad()                          # 1. a method of the optimiser: forget the old gradient
    toy_loss.backward()                                # 2. a method of the loss: its gradient goes to knob.grad
    toy_optimiser.step()                               # 3. a method of the optimiser: knob = knob - 0.1 * knob.grad
    print("update %d: loss %.2f, gradient %.2f, the knob moved to %.3f" % (update, toy_loss.item(), knob.grad.item(), knob.item()))

**What to write.** `reinforce_step(optimiser, loss)` gets the optimiser and the loss of this update as its two arguments, so the three methods are called on them:

- **3a)** a method of the optimiser: forget the old gradients;
- **3b)** a method of the loss: compute the new gradient;
- **3c)** a method of the optimiser: take the step.

🎯 **Piece 3.** Complete the three lines marked 3a, 3b and 3c, in this order.

<details><summary>💡 Hint</summary>

The cell above did it on the toy: `zero_grad()` and `step()` are called on the optimiser, and `backward()` is called on the loss. Here the optimiser and the loss are the two arguments of the function.
</details>

In [ ]:
def reinforce_step(optimiser, loss):
    ...                         # 🎯 3a) forget the gradients of the previous update
    ...                         # 🎯 3b) autograd: the gradient of the loss with respect to w
    ...                         # 🎯 3c) move w one step, downhill on the loss

In [ ]:
check_reinforce_step(reinforce_step)

## Piece 4: the coins to go

So far every road of a day was pushed by the coins of the whole day. The popular implementations are more careful. A road cannot be responsible for the coins that Mara collected before she took it, so they push each road by the coins collected **from that road on**. The code calls this the return, or the reward to go.

Take a day that paid 7, 7 and 7. The first road is pushed by 21, the second by 14 (the last two arrivals) and the third by 7 (the last arrival). A day that paid 9, 4 and 2 gives 15, 6 and 2.

`reward_to_go(coins)` gets the coins of a batch of days, a tensor `coins` of shape `(B, 3)` with one row for every day and one column for every arrival (the first column holds the first arrival, and so on), and returns a tensor of the same shape.

### The tools

Two tensor methods give the coins to go. The cell below shows each of them alone, on two days with made-up numbers.

- `x.flip(1)` reverses the order of the numbers in every row. The 1 is the direction, the same as `dim=1`: along each row.
- `x.cumsum(dim=1)` gives the running total along every row: the first number, then the first plus the second, then the first plus the second plus the third. Every entry of the result is the sum of the entries of its row up to that position.

Put them together: read the day backwards, take the running total, and read the totals in the order of the day again. For the day 9, 4, 2 the backwards day is 2, 4, 9, its running totals are 2, 6, 15, and in the order of the day they are 15, 6, 2. Every running total of the backwards day is the sum of the coins from some road to the end of the day, which is exactly the coins to go.

In [ ]:
# Two days (the rows) of three arrivals each (the columns).
x = torch.tensor([[1., 2., 3.],
                  [10., 20., 30.]])
print("x.flip(1) reverses every row:")
print(x.flip(1))
print()
print("x.cumsum(dim=1) is the running total along every row:")
print(x.cumsum(dim=1))

**What to write.** In `reward_to_go`, `coins` has the shape `(B, 3)` as in the cell above:

- **4a)** `backwards`: `coins` with every row reversed, with `.flip(1)`;
- **4b)** the value to return: the running totals of `backwards` along each row, with `.cumsum(dim=1)`, and then reversed again with `.flip(1)`, so that the numbers are back in the order of the day.

🎯 **Piece 4.** Complete the two lines marked 4a and 4b.

<details><summary>💡 Hint</summary>

Methods can be chained: `backwards.cumsum(dim=1)` is a tensor, so you can call `.flip(1)` on it. For one day, the coins 9, 4, 2 become 2, 4, 9 after line 4a, their running totals are 2, 6, 15, and reversed again they are 15, 6, 2.
</details>

In [ ]:
def reward_to_go(coins):
    # coins has shape (B, 3). The answer has the same shape: for every road, the coins collected from that road to the end of the day.
    backwards = ...                                     # 🎯 4a) read every day backwards, from its last arrival to its first
    return ...                                          # 🎯 4b) running totals of the backwards day, flipped back into the order of the day

In [ ]:
check_reward_to_go(reward_to_go)

## The training loop

Nothing to fill in. `train` ties your pieces together: a fresh table of knobs, then `updates` rounds of play `B` days, build the pseudo-loss, take a step. After every update it records the exact average coins of the table, for the plot only.

`make_multiplier` builds the multiplier of every road. By default it is the coins of the whole day. The three switches of `train` (`to_go`, `standardise` and `adam`) are for the section after the first run.

In [ ]:
def make_multiplier(days, to_go=False, standardise=False):
    # The number that multiplies the score of every road. One per road, shape (B, 3).
    if to_go:
        multiplier = reward_to_go(days.coins)                              # Piece 4: the coins from each road on
    else:
        G = days.coins.sum(dim=1, keepdim=True)                            # the coins of each whole day, shape (B, 1)
        multiplier = G.expand(-1, 3)                                       # the same G for the three roads of the day
    if standardise == "day":
        # the line of the PyTorch example, each day on its own: its three numbers get average 0 and spread 1
        multiplier = (multiplier - multiplier.mean(dim=1, keepdim=True)) / (multiplier.std(dim=1, keepdim=True) + TINY)
    elif standardise:
        # the same line over all the numbers of the batch: average 0 and spread 1
        multiplier = (multiplier - multiplier.mean()) / (multiplier.std() + TINY)
    return multiplier


def train(updates, B, alpha, seed, to_go=False, standardise=False, adam=False):
    torch.manual_seed(seed)                                                # the dice of this run
    w = torch.zeros(2, 7, requires_grad=True)                              # the table of knobs, every knob at 0
    optimiser = torch.optim.Adam([w], lr=alpha) if adam else torch.optim.SGD([w], lr=alpha)
    curve = [exact_J(w)]                                                   # the exact average coins before the first update
    for update in range(updates):
        days = play_days(w, B)                                             # play B days with the current table
        multiplier = make_multiplier(days, to_go, standardise)
        reinforce_step(optimiser, pseudo_loss(days, multiplier))           # Pieces 2 and 3: one update
        curve.append(exact_J(w))                                           # for the plot only: the method never sees it
    return w.detach(), curve

## Run it and look

Nothing to fill in. Every run starts with every knob at 0. A seed fixes all the dice of a run, so a run can be repeated. The cell plays five seeds. The numbers below are yours to change: run the cell again after a change.

In [ ]:
B = 100            # days played in every update
ALPHA = 0.1        # step size: our alpha, PyTorch's lr
UPDATES = 100      # how many updates
SEEDS = 5          # how many runs (seeds 0, 1, 2, ...)

if pieces_ready(1, 2, 3):
    plain_runs = [train(UPDATES, B, ALPHA, seed) for seed in range(SEEDS)]
    show_runs(plain_runs, B)

## What you just saw

The grasshopper climbed again. This time one update can move all fourteen knobs, from the same 100 days. Nobody told Mara what a place pays: she saw the coins of her days, and that was enough.

Two things to notice in the picture. The curves wobble, because every update uses a noisy estimate. And the table at the end gives most of its chance to the three roads of the best path.

Each run played 10,000 days. The next section makes that much cheaper.

## What the popular implementations add

The code you wrote is the core of REINFORCE. The popular implementations add a few lines, and each one is a switch in `train`. Turn them on one at a time in the cell below and run it again. Run the cell of the first run before it, because the grey curves are those runs.

1. `TO_GO = True` pushes each road by the coins from that road on. That is your Piece 4, and the PyTorch example does it with its `returns`.
2. `STANDARDISE = True` subtracts the average of the multipliers and divides by their spread. In the PyTorch example it is one line: `(returns - returns.mean()) / (returns.std() + eps)`. Here the line runs over all the multipliers of the batch.
3. `ADAM = True` uses Adam in place of plain gradient steps, as the PyTorch example and Spinning Up do.

Each switch changes the size of the multipliers or the way the steps are taken, so each combination needs its own step size. The table under the cell lists the best one for each. The cell draws your run in blue and the first run in grey, with a logarithmic axis for the days.

In [ ]:
TO_GO = False          # switch 1: push each road by the coins from that road on (your Piece 4)
STANDARDISE = False    # switch 2: True standardises the multipliers over the batch, "day" does it day by day
ADAM = False           # switch 3: Adam in place of plain gradient steps
ALPHA = 0.1            # the step size that goes with the switches: the table below the cell lists the best ones

if pieces_ready(1, 2, 3) and (not TO_GO or pieces_ready(4)):
    runs = [train(UPDATES, B, ALPHA, seed, TO_GO, STANDARDISE, ADAM) for seed in range(SEEDS)]
    show_runs(runs, B, base=plain_runs, logx=True)

### What the switches did on this road

Each row is 100 runs (seeds 0 to 99) as long as the first run: 100 updates of 100 days. The step size of a row comes from separate tuning runs, 200 runs for every step size on a grid an eighth of a decade apart: the largest step at which every run found the best path, halved and rounded to 1, 2, 3 or 5 times a power of ten. A run is a success when the best path is the most likely path at the end.

| switches on | step size | runs that found the best path | days until 90% of the gap is closed (median run) |
|--|--|--|--|
| none: the coins of the whole day, plain steps | 0.1 | 100 of 100 | 6,500 |
| `TO_GO` | 0.2 | 100 of 100 | 3,100 |
| `TO_GO`, `STANDARDISE` | 10 | 100 of 100 | 400 |
| `TO_GO`, `STANDARDISE`, `ADAM` | 0.3 | 100 of 100 | 600 |

- **Reward to go** saves about half of the days. A road is no longer pushed by coins that were collected before it.
- **Standardising is the switch that matters on this road.** The multipliers become small numbers around 0 in place of coins that are all positive, the step size can grow from 0.2 to 10, and the days fall from 3,100 to 400. Try `TO_GO = False` as well, with the same step size: standardising alone needs 200 days.
- **Adam brings nothing on this road.** With a tuned learning rate it needs 600 days against 400 for plain steps. With the popular learning rate 0.01 it needs about 21,700 days, because Adam moves each knob by about its learning rate in an update, and these knobs have to travel a few units.
- **The PyTorch example standardises one episode at a time.** On a day of three roads that line breaks. Run it with `TO_GO = True` and `STANDARDISE = "day"`: none of the 100 runs found the best path, at either of two step sizes, and the average coins fell from 17.25 to 16.7 with the step size 0.1 and to 14.4 with the step size 10. The coins to go shrink along a day, so after standardising the first road of every day is pushed up and the last road is pushed down, whatever the day paid. A line that works on CartPole needs checking on a new problem.

The next part of the lecture explains why subtracting the average steadies the estimate so much, and why a steadier estimate allows a much larger step.

## ⭐ Optional: what a larger step does

The step size 0.1 of the first run was chosen with care. A larger step is faster at first, and it has a price. All the coins are positive, so every road that Mara takes is pushed up. When the push is large, the roads of the first days become so likely that she stops trying the others, and the table can lock onto a path that is not the best one.

The cell runs 20 seeds with a step ten times larger. Some end on the best path. The others end on a worse path, usually one of the two that pay 19 coins on average.

In [ ]:
ALPHA_BIG = 1.0     # a step ten times larger than the first run's

if pieces_ready(1, 2, 3):
    big_runs = [train(60, 100, ALPHA_BIG, seed) for seed in range(20)]    # 20 seeds, 60 updates of 100 days
    show_lock_in(big_runs, 100)

## Mara's road, learned

The last cell runs REINFORCE with the two switches that mattered, `TO_GO` and `STANDARDISE`, with the step size from the table, on 20 seeds of 20 updates of 100 days. It draws the table of chances of the first seed on the road. Every road shows its chance, and the three roads of the best path are outlined.

In [ ]:
if pieces_ready(1, 2, 3, 4):
    final_runs = [train(20, 100, 10.0, seed, to_go=True, standardise=True) for seed in range(20)]
    show_runs(final_runs, 100, logx=True)

## What it cost

The first exercise played 200 days in every iteration to measure the slope of one knob. With the settings of that notebook a run played 60,000 days, and the average coins closed 90% of the gap between the start and the best path after about 35,000 of them. The first run of this notebook played 10,000 days and closed 90% of the gap after about 6,500, with every day teaching all fourteen knobs. With the two switches of the closing cell it takes about 400 days. That is about 90 times fewer days than the grasshopper's method needed for the same climb.

A word of caution. The first exercise's method can be tuned to play much less: 750 days, which find the best path in 99 of 100 runs, although its average coins then stop between 19.7 and 20.5. And on a town with eight paths, simply trying every path a few times is cheaper than any of these methods. Play each path 7 times, 56 days in all, and the path with the best average is the best path in 99 tries out of 100. A table this small needs neither REINFORCE nor the grasshopper's trick. They are built for problems with far too many paths to try.